In [ ]:
from run_single_experiment import run_experiment_1, run_experiment_2, run_experiment_3, run_experiment_4

In [ ]:
from pathlib import Path

RESULTS_BASE = Path("./results")
ERROR_BASE = Path("./error_analysis")

def _norm_model_name(model: str) -> str:
    return model.replace("/", "_")

def exp_results_dir(exp_name: str, provider: str, model: str) -> str:
    path = RESULTS_BASE / exp_name / provider.lower() / _norm_model_name(model)
    path.mkdir(parents=True, exist_ok=True)
    return str(path)

def exp_out_csv(exp_name: str, provider: str, model: str, filename: str = "results.csv") -> str:
    return str(Path(exp_results_dir(exp_name, provider, model)) / filename)

def exp_error_dir(exp_name: str, provider: str, model: str) -> str:
    path = ERROR_BASE / exp_name / provider.lower() / _norm_model_name(model)
    path.mkdir(parents=True, exist_ok=True)
    return str(path)


In [ ]:
SUPPORTED_TYPES = {
    "Dice_Count",
    "Geometry_Click",
    "Image_Matching",
    "Patch_Select",
    "Place_Dot",

    "Bingo",
    "Click_Order",
    "Dart_Count",
    "Image_Recognition",
    "Misleading_Click",
    "Object_Match",
    "Path_Finder",
    "Pick_Area",
    "Select_Animal",
    "Unusual_Detection",
    "Coordinates",
    "Connect_Icon",
    "Rotation_Match",
}

ERROR_TYPES = {
    "Patch_Select", 
    "Click_Order", 
    "Pick_Area", 
    "Place_Dot", 
    "Dice_Count"}

In [ ]:
# Experiment 1: Ground Truth Prompts
provider = "gemini"
model = "gemini-2.5-flash"
reasoning_effort = None
storage_model = model if reasoning_effort is None else f"{model}_{reasoning_effort}"
thinking_opts = None if reasoning_effort is None else {"effort": reasoning_effort}

result1 = run_experiment_1(
    types=SUPPORTED_TYPES,
    max_per_type=20,
    provider=provider,
    model=model,
    out_csv=exp_out_csv("exp1", provider, storage_model),
    thinking=False,
    thinking_options=thinking_opts,
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp1", provider, storage_model),
    enable_error_analysis=False,
    error_analysis_dir=exp_error_dir("exp1", provider, storage_model),
    collect_reasoning=False,
    timeout_sec=600
)


In [ ]:
# Experiment 2: Optimized Prompts
provider = "openai"
model = "gpt-5.1"
reasoning_effort = "medium"
storage_model = model if reasoning_effort is None else f"{model}_{reasoning_effort}"
thinking_opts = None if reasoning_effort is None else {"effort": reasoning_effort}

result2 = run_experiment_2(
    types=["Patch_Select"],
    prompts_file="./prompts_optimized.yaml",
    max_per_type=20,
    provider=provider,
    model=model,
    out_csv=exp_out_csv("exp2", provider, storage_model),
    thinking=False,
    thinking_options=thinking_opts,
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp2", provider, storage_model),
    enable_error_analysis=False,
    error_analysis_dir=exp_error_dir("exp2", provider, storage_model),
    collect_reasoning=False,
    timeout_sec=600
)

In [ ]:
# Experiment 3: Until Correct
provider = "openai"
model = "gpt-5.1"
reasoning_effort = "none"
storage_model = model if reasoning_effort is None else f"{model}_{reasoning_effort}"
thinking_opts = None if reasoning_effort is None else {"effort": reasoning_effort}

result3 = run_experiment_3(
    types=SUPPORTED_TYPES,
    prompts_file="./prompts_optimized.yaml",
    prompt_mode="auto",
    max_attempts_per_type=10,
    max_pool_per_type=20,
    use_full_dataset_pool=False,
    out_csv=exp_out_csv("exp3", provider, model),
    provider=provider,
    model=model,
    thinking=False,
    thinking_options=None,
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp3", provider, model),
    collect_reasoning=False
)


In [ ]:
# Experiment 4: Few-shot
provider = "openai"
model = "gpt-5.1"
reasoning_effort = "medium"
storage_model = model if reasoning_effort is None else f"{model}_{reasoning_effort}"
thinking_opts = None if reasoning_effort is None else {"effort": reasoning_effort}

result4 = run_experiment_4(
    prompts_file="./prompts_optimized.yaml",
    few_shot_file="./few_shot_examples.yaml",
    types=ERROR_TYPES,
    max_per_type=10,
    provider=provider,
    model=model,
    n_shot=2,
    thinking=False,
    thinking_options={"effort": reasoning_effort},
    enable_error_analysis=False,
    error_analysis_dir=exp_error_dir("exp4", provider, model),
    collect_tokens=True,
    token_output_dir=exp_results_dir("exp4", provider, model),
    collect_reasoning=False,
    out_csv=exp_out_csv("exp4", provider, model),
    timeout_sec=600
)


In [ ]:
# Error Analysis
provider = "openai"
model = "gpt-5"
reasoning_effort = "medium"
storage_model = model if reasoning_effort is None else f"{model}_{reasoning_effort}"
thinking_opts = None if reasoning_effort is None else {"effort": reasoning_effort}

result4 = run_experiment_4(
    prompts_file="./prompts_optimized.yaml",
    few_shot_file="./few_shot_examples.yaml",
    types=ERROR_TYPES,
    max_per_type=4,
    provider=provider,
    model=model,
    n_shot=2,
    thinking=True,
    thinking_options={"effort": reasoning_effort},
    enable_error_analysis=True,
    error_analysis_dir=exp_error_dir("exp4", provider, model),
    collect_tokens=True,
    token_output_dir=exp_results_dir("error_analysis", provider, model),
    collect_reasoning=True,
    out_csv=exp_out_csv("error_analysis", provider, model),
    timeout_sec=600
)

In [ ]:
import pandas as pd
from pathlib import Path

# Task types that need updating
TYPES_TO_UPDATE = ["Misleading_Click", "Pick_Area", "Object_Match"]

def update_exp2_for_types(
    provider: str,
    storage_model: str,
    api_model: str | None = None,
    types: list[str] | None = None,
    prompts_file: str = "./prompts_optimized.yaml",
    max_per_type: int | None = None,
    thinking: bool = False,
    thinking_options: dict | None = None,
):
    """
    Re-run Exp2 for specified task types only and merge results back to existing results.csv.
    
    Args:
        provider: "gemini" / "openai" / "anthropic" / "fireworks"
        storage_model: Model name used in results directory (e.g., "gemini-2.5-flash" or "gpt-5.1_medium")
        api_model: Model name for API calls, defaults to storage_model (e.g., "gpt-5.1" when storage is "gpt-5.1_medium")
        types: List of task types to update, defaults to TYPES_TO_UPDATE
        prompts_file: Path to optimized prompts file
        max_per_type: Max questions per type, inferred from existing results if None
        thinking: Whether to enable thinking mode
        thinking_options: Thinking configuration options
    """
    if types is None:
        types = TYPES_TO_UPDATE

    api_model = api_model or storage_model

    # Main result file & temporary file
    main_csv = Path(exp_out_csv("exp2", provider, storage_model, filename="results.csv"))
    tmp_csv  = Path(exp_out_csv("exp2", provider, storage_model, filename="results_tmp.csv"))

    if not main_csv.exists():
        raise FileNotFoundError(f"Main results file not found: {main_csv}")

    # Read old results to infer max_per_type (optional)
    old = pd.read_csv(main_csv)
    col_type = "type" if "type" in old.columns else "task_type"

    if max_per_type is None:
        n_map = {}
        for t in types:
            rows_t = old[old[col_type] == t]
            if not rows_t.empty and "n" in rows_t.columns:
                n_map[t] = int(rows_t["n"].max())
        max_per_type = max(n_map.values()) if n_map else 15

    print(f"[INFO] Re-running Exp2: provider={provider}, api_model={api_model}, "
          f"storage_model={storage_model}, types={types}, max_per_type={max_per_type}")

    # 1. Re-run Exp2 for specified types only, output to temp CSV
    _= run_experiment_2(
        dataset_root="./captcha_data",
        types=types,
        provider=provider,
        model=api_model,
        max_per_type=max_per_type,
        secrets_file="./secrets.yaml",
        prompts_file=prompts_file,
        out_csv=str(tmp_csv),
        thinking=thinking,
        thinking_options=thinking_options,
        enable_error_analysis=False,
        error_analysis_dir=None,
        collect_tokens=False,
        token_output_dir=None,
        collect_reasoning=False,
        timeout_sec=600.0,
    )

    # 2. Read new results
    new = pd.read_csv(tmp_csv)
    # Align column names
    if "task_type" in new.columns and col_type == "type":
        new = new.rename(columns={"task_type": "type"})
    elif "type" in new.columns and col_type == "task_type":
        new = new.rename(columns={"type": "task_type"})

    # 3. Backup old file
    backup = main_csv.with_suffix(main_csv.suffix + ".bak")
    old.to_csv(backup, index=False)

    # 4. Remove old rows for these types and append new results
    updated = pd.concat(
        [
            old[~old[col_type].isin(types)],
            new
        ],
        ignore_index=True
    )
    updated.to_csv(main_csv, index=False)

    print(f"[DONE] Updated {main_csv} for {types}; backup saved as {backup}")


# Example usage:
# 1) Gemini 2.5 Flash (directory name matches API model)
# update_exp2_for_types("gemini", "gemini-2.5-flash")

# 2) OpenAI GPT-5.1 (medium): directory name is gpt-5.1_medium, but API model is gpt-5.1
# update_exp2_for_types("openai", "gpt-5.1_medium", api_model="gpt-5.1")


In [ ]:
update_exp2_for_types(
    provider="openai",
    storage_model="gpt-5.1_medium",
    api_model="gpt-5.1",
    thinking=True,
    thinking_options={"effort": "medium"},
)
